# Web Scraping & Product Price Analysis – Air Conditioners (Amazon India)
**Stage 5 – Data Aggregation & Representation** (simplified, beginner version)

Pipeline position: **5 of 8**

## Objective
Summarise the cleaned data with `groupby()`, `pivot_table()` and `crosstab()`, and save each summary table as a CSV.

## Step 1 – Setup

In [9]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Folder that holds all the CSV files of the project.
# (In Google Colab: upload the CSV files and change this to "/content/")
DATA = "../0. DataSet/"

print("Setup complete.")

Setup complete.


In [10]:
import os

df = pd.read_csv("Air_Conditioners_Cleaned.csv")
print("Loaded:", df.shape)

# folder where the summary tables will be saved
os.makedirs("aggregates", exist_ok=True)

Loaded: (652, 15)


## Step 2 – Overall summary

In [11]:
overall = pd.DataFrame({
    "Listings": [len(df)],
    "Listings with price": [df["discount_price"].notna().sum()],
    "Average price (Rs)": [df["discount_price"].mean()],
    "Median price (Rs)": [df["discount_price"].median()],
    "Average discount %": [df["discount_percent"].mean()],
    "Average rating": [df["ratings"].mean()],
    "Number of brands": [df["brand"].nunique()],
}).round(2)
overall.T

,0
Listings,652.00
Listings with price,423.00
Average price (Rs),43102.61
Median price (Rs),39990.00
Average discount %,29.10
Average rating,3.87
Number of brands,22.00


## Step 3 – Group by brand
`groupby("brand")` splits the rows into one group per brand. Then we calculate something for each group (count, mean ...).

In [12]:
by_brand = df.groupby("brand")[["discount_price", "discount_percent", "ratings"]].mean().round(2)
by_brand["listings"] = df.groupby("brand")["name"].count()
by_brand = by_brand.sort_values("listings", ascending=False)
by_brand

,discount_price,discount_percent,ratings,listings
brand,,,,
Other,41173.76,26.73,4.02,103
Voltas,44719.30,33.29,3.96,82
LG,43787.81,32.53,4.05,65
Daikin,47851.09,25.50,3.97,50
Samsung,42034.24,29.23,3.52,41
Carrier,41162.62,30.53,3.72,41
Blue Star,40418.63,26.70,3.97,38
Panasonic,45414.00,28.44,4.23,36
Lloyd,34771.71,41.00,3.99,30


## Step 4 – Group by tonnage, energy star, AC type, inverter and Wi-Fi
The same idea, with a different column each time.

In [13]:
by_tonnage = df.groupby("tonnage")[["discount_price", "ratings"]].mean().round(2)
by_tonnage["listings"] = df.groupby("tonnage")["name"].count()
by_tonnage

,discount_price,ratings,listings
tonnage,,,
0.75000,28180.00,3.60,1
0.80000,27990.00,3.90,4
1.00000,36100.64,3.90,160
1.10000,42994.50,4.30,3
1.20000,33723.00,3.80,3
1.25000,27999.00,4.00,1
1.40000,34994.50,4.15,2
1.50000,40639.10,3.89,342
1.51066,52600.00,NaN,1


In [14]:
by_star = df.groupby("energy_star")[["discount_price", "ratings"]].mean().round(2)
by_star["listings"] = df.groupby("energy_star")["name"].count()
by_star

,discount_price,ratings,listings
energy_star,,,
1.0,47046.67,3.50,6
2.0,38632.55,3.61,18
3.0,40768.68,3.84,362
4.0,42632.65,3.85,53
5.0,46016.06,3.99,162


In [15]:
# Let's print the available columns to find the correct names from Stage 3
print("Available columns:", list(df.columns))

# Safely group by 'ac_type' if it exists, or look for alternative names
ac_type_col = "ac_type" if "ac_type" in df.columns else ("type" if "type" in df.columns else None)
is_inverter_col = "is_inverter" if "is_inverter" in df.columns else ("inverter" if "inverter" in df.columns else None)
has_wifi_col = "has_wifi" if "has_wifi" in df.columns else ("wifi" if "wifi" in df.columns else None)

if ac_type_col:
    by_type = df.groupby(ac_type_col)[["discount_price", "ratings"]].mean().round(2)
    by_type["listings"] = df.groupby(ac_type_col)["name"].count()
    print("\nBy AC type:")
    print(by_type)
else:
    print("\nWarning: AC type column not found.")
    by_type = pd.DataFrame() # Empty fallback

if is_inverter_col:
    by_inverter = df.groupby(is_inverter_col)[["discount_price", "ratings"]].mean().round(2)
    by_inverter["listings"] = df.groupby(is_inverter_col)["name"].count()
    print("\nInverter (True) vs non-inverter (False):")
    print(by_inverter)
else:
    print("\nWarning: Inverter column not found.")
    by_inverter = pd.DataFrame()

if has_wifi_col:
    by_wifi = df.groupby(has_wifi_col)[["discount_price", "ratings"]].mean().round(2)
    by_wifi["listings"] = df.groupby(has_wifi_col)["name"].count()
    print("\nWi-Fi (True) vs no Wi-Fi (False):")
    print(by_wifi)
else:
    print("\nWarning: Wi-Fi column not found.")
    by_wifi = pd.DataFrame()

Available columns: ['name', 'main_category', 'sub_category', 'image', 'link', 'ratings', 'no_of_ratings', 'discount_price', 'actual_price', 'discount_amount', 'discount_percent', 'tonnage', 'energy_star', 'is_ac', 'brand']





## Step 5 – Price bands
`pd.cut()` puts each price into a band (bin). The band edges are listed in `bins`; `labels` are the band names.

In [16]:
bins = [0, 30000, 40000, 50000, 1000000]
labels = ["Budget (< 30k)", "Mid (30k-40k)", "Upper-mid (40k-50k)", "Premium (50k+)"]
df["price_band"] = pd.cut(df["discount_price"], bins=bins, labels=labels, right=False)

price_bands = df.groupby("price_band", observed=False)[["ratings", "discount_percent"]].mean().round(2)
price_bands["listings"] = df.groupby("price_band", observed=False)["name"].count()
price_bands["share_%"] = (price_bands["listings"] / price_bands["listings"].sum() * 100).round(1)
price_bands

,ratings,discount_percent,listings,share_%
price_band,,,,
Budget (< 30k),3.92,31.37,30,7.1
Mid (30k-40k),3.95,32.99,187,44.2
Upper-mid (40k-50k),3.91,28.33,122,28.8
Premium (50k+),3.79,20.76,84,19.9


## Step 6 – Pivot table and crosstab
A **pivot table** shows an average in a grid (rows × columns). A **crosstab** counts how many listings fall in each cell.

In [17]:
pivot_price = pd.pivot_table(df, values="discount_price", index="energy_star", columns="tonnage", aggfunc="mean").round(0)
pivot_price

tonnage,0.75000,0.80000,1.00000,1.10000,1.20000,1.25000,1.40000,1.50000,1.51066,1.60000,1.70000,1.80000,2.00000,2.02000,2.10000,2.20000,6.00000
energy_star,,,,,,,,,,,,,,,,,
1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,44290.0,48425.0,NaN,NaN,NaN,NaN
2.0,NaN,NaN,27826.0,NaN,NaN,NaN,NaN,34293.0,NaN,NaN,NaN,46900.0,47925.0,NaN,NaN,NaN,NaN
3.0,NaN,27990.0,34650.0,42994.0,33723.0,27999.0,31990.0,38859.0,52600.0,58400.0,NaN,54514.0,54015.0,61480.0,66000.0,79350.0,67900.0
4.0,NaN,NaN,35184.0,NaN,NaN,NaN,NaN,39301.0,NaN,NaN,49150.0,NaN,53895.0,65890.0,NaN,69990.0,NaN
5.0,28180.0,NaN,40274.0,NaN,NaN,NaN,37999.0,44937.0,NaN,43745.0,NaN,67297.0,63701.0,NaN,NaN,NaN,NaN


In [18]:
crosstab_brand_star = pd.crosstab(df["brand"], df["energy_star"])
crosstab_brand_star

energy_star,1.0,2.0,3.0,4.0,5.0
brand,,,,,
Blue Star,0,3,24,0,9
Carrier,0,1,28,1,11
Daikin,1,1,22,8,15
Electrolux,0,0,3,0,2
Godrej,0,0,15,1,4
Haier,1,0,17,4,6
Hitachi,0,2,10,0,4
IFB,0,0,2,0,0
Kelvinator,0,0,1,0,0


## Step 7 – Ranking brands
`rank()` gives 1 to the biggest value.

In [19]:
rank = by_brand[["listings", "discount_price", "ratings"]].copy()
rank["price_rank"] = rank["discount_price"].rank(ascending=False)
rank["rating_rank"] = rank["ratings"].rank(ascending=False)
rank.sort_values("price_rank")

,listings,discount_price,ratings,price_rank,rating_rank
brand,,,,,
Mitsubishi,5,66199.80,3.68,1.0,14.5
O General,25,58168.64,3.58,2.0,16.0
Daikin,50,47851.09,3.97,3.0,8.5
Toshiba,9,47710.33,2.90,4.0,19.0
Panasonic,36,45414.00,4.23,5.0,3.0
Voltas,82,44719.30,3.96,6.0,10.0
Electrolux,5,44190.00,4.45,7.0,1.0
LG,65,43787.81,4.05,8.0,4.0
IFB,2,42547.50,3.70,9.0,13.0


## Step 8 – Save all tables

In [20]:
by_brand.to_csv( "aggregates/by_brand.csv")
by_tonnage.to_csv(  "aggregates/by_tonnage.csv")
by_star.to_csv(  "aggregates/by_star.csv")
by_type.to_csv( "aggregates/by_type.csv")
by_inverter.to_csv(  "aggregates/by_inverter.csv")
by_wifi.to_csv( "aggregates/by_wifi.csv")
price_bands.to_csv(  "aggregates/price_bands.csv")
overall.to_csv( "aggregates/overall.csv")
pivot_price.to_csv(  "aggregates/pivot_price_star_x_tonnage.csv")
crosstab_brand_star.to_csv(  "aggregates/crosstab_brand_x_star.csv")

print("Saved files:", sorted(os.listdir( "aggregates")))

Saved files: ['by_brand.csv', 'by_inverter.csv', 'by_star.csv', 'by_tonnage.csv', 'by_type.csv', 'by_wifi.csv', 'crosstab_brand_x_star.csv', 'overall.csv', 'pivot_price_star_x_tonnage.csv', 'price_bands.csv']


**Output of Stage 5:** summary tables in `0. DataSet/aggregates/`.  **Next → Stage 6: Data Analysis.**